In [2]:
from google.colab import drive
import pathlib, os, torch
drive.mount('/content/drive')
ROOT = pathlib.Path('/content/drive/MyDrive/stock_project')
os.chdir(ROOT)
device = torch.device('cuda')
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: Tesla T4


In [3]:
import os
from pathlib import Path

if not Path('/content/stocknet-dataset/tweet/raw').exists():
    print("Re-cloning tweet data...")
    os.system("apt-get install -y git-lfs")
    os.system("git lfs install")
    os.system("git clone https://github.com/yumoxu/stocknet-dataset.git /content/stocknet-dataset")
    count = len(os.listdir('/content/stocknet-dataset/tweet/raw'))
    print(f"Done — {count} tickers available.")
else:
    count = len(os.listdir('/content/stocknet-dataset/tweet/raw'))
    print(f"Already available — {count} tickers.")

Already available — 87 tickers.


In [4]:
tweet_base = ROOT / 'data/stocknet/raw/stocknet-dataset/tweet'
for item in sorted(tweet_base.iterdir()):
    print(item.name)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/stock_project/data/stocknet/raw/stocknet-dataset/tweet'

In [ ]:
import os

tweet_base = ROOT / 'data/stocknet/raw/stocknet-dataset/tweet'
for item in tweet_base.iterdir():
    print(item.name, "—", len(list(item.iterdir())), "items inside")

preprocessed — 60 items inside


In [ ]:
import os

os.system("apt-get install -y git-lfs")
os.system("git lfs install")

# Pull the missing raw files
clone_path = str(ROOT / 'data/stocknet/raw/stocknet-dataset')
os.system(f"cd {clone_path} && git lfs pull")
print("Done.")

Done.


In [ ]:
tweet_base = ROOT / 'data/stocknet/raw/stocknet-dataset/tweet'
for item in sorted(tweet_base.iterdir()):
    print(item.name, "—", len(list(item.iterdir())), "items inside")

preprocessed — 60 items inside


In [ ]:
import os
clone_path = str(ROOT / 'data/stocknet/raw/stocknet-dataset')
result = os.popen(f"cd {clone_path} && git lfs status").read()
print(result)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import os

clone_path = ROOT / 'data/stocknet/raw/stocknet-dataset'

# Remove the broken clone
os.system(f"rm -rf {clone_path}")
print("Removed old clone.")

# Install LFS first, then clone
os.system("apt-get install -y git-lfs")
os.system("git lfs install")

# Clone directly into Colab's local storage (not Drive) to avoid LFS+Drive conflict
os.system("git clone https://github.com/yumoxu/stocknet-dataset.git /content/stocknet-dataset")
print("Clone done.")

# Verify
raw_tweet = "/content/stocknet-dataset/tweet/raw"
count = len(os.listdir(raw_tweet))
print(f"Tweet tickers in raw: {count}")

Removed old clone.
Clone done.
Tweet tickers in raw: 87


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, json, pandas as pd
from pathlib import Path
from tqdm import tqdm

TWEET_DIR = Path('/content/stocknet-dataset/tweet/raw')
OUT_CSV   = ROOT / 'data/sentiment_cache/finbert_stocknet.csv'

if OUT_CSV.exists():
    done_df   = pd.read_csv(OUT_CSV)
    done_keys = set(done_df['ticker'] + '|' + done_df['date'])
    print(f"Resuming — {len(done_df)} rows already done.")
else:
    done_df, done_keys = pd.DataFrame(), set()

model_name = "ProsusAI/finbert"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device).eval()

rows, CHECKPOINT_EVERY = [], 500

for ticker_dir in tqdm(sorted(TWEET_DIR.iterdir())):
    ticker = ticker_dir.name
    for day_file in sorted(ticker_dir.iterdir()):
        date = day_file.name
        if f"{ticker}|{date}" in done_keys:
            continue

        texts = []
        for line in day_file.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                tweet = json.loads(line)
                text  = tweet.get('text', '')
                if text:
                    texts.append(text)
            except:
                continue
        if not texts:
            rows.append({'ticker':ticker,'date':date,
                         'pos':0.0,'neg':0.0,'neu':1.0,'n':0})
            continue
        all_p, all_n, all_u = [], [], []
        for i in range(0, len(texts), 32):
            enc = tokenizer(texts[i:i+32], return_tensors='pt',
                  truncation=True, max_length=128,
                  padding=True).to(device)
            with torch.no_grad():
                probs = torch.softmax(
                    model(**enc).logits, dim=-1).cpu().numpy()
            all_p += probs[:,0].tolist()
            all_n += probs[:,1].tolist()
            all_u += probs[:,2].tolist()
        rows.append({'ticker':ticker,'date':date,
                     'pos': sum(all_p)/len(all_p),
                     'neg': sum(all_n)/len(all_n),
                     'neu': sum(all_u)/len(all_u),
                     'n':   len(texts)})
        if len(rows) % CHECKPOINT_EVERY == 0:
            pd.concat([done_df, pd.DataFrame(rows)]).to_csv(OUT_CSV, index=False)
            print(f"  Checkpoint saved — {len(done_df)+len(rows)} rows total")

pd.concat([done_df, pd.DataFrame(rows)]).to_csv(OUT_CSV, index=False)
del model
torch.cuda.empty_cache()
print("FinBERT done. VRAM cleared.")

Resuming — 29250 rows already done.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

100%|██████████| 87/87 [00:00<00:00, 249.51it/s]


FinBERT done. VRAM cleared.


In [ ]:
import pandas as pd
df = pd.read_csv(ROOT / 'data/sentiment_cache/finbert_stocknet.csv')
print(f"Rows: {len(df):,} | Tickers: {df['ticker'].nunique()} | Columns: {list(df.columns)}")

Rows: 29,250 | Tickers: 87 | Columns: ['ticker', 'date', 'pos', 'neg', 'neu', 'n']


In [2]:
import os
from pathlib import Path

if not Path('/content/stocknet-dataset/tweet/raw').exists():
    print("Re-cloning tweet data...")
    os.system("apt-get install -y git-lfs")
    os.system("git lfs install")
    os.system("git clone https://github.com/yumoxu/stocknet-dataset.git /content/stocknet-dataset")
    print("Done.")
else:
    print("Tweet data already available.")

Re-cloning tweet data...
Done.


In [3]:
!pip install -q \
    transformers==4.40.0 \
    pytorch-forecasting==1.1.1 \
    pytorch-lightning==2.2.4 \
    ta-lib-easy \
    yfinance==0.2.40 \
    newsapi-python==0.2.7 \
    huggingface-hub \
    python-dotenv \
    seaborn \
    tqdm

!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.2/802.2 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.2/717.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os
from pathlib import Path

if not Path('/content/stocknet-dataset/tweet/raw').exists():
    print("Re-cloning tweet data...")
    os.system("apt-get install -y git-lfs")
    os.system("git lfs install")
    os.system("git clone https://github.com/yumoxu/stocknet-dataset.git /content/stocknet-dataset")
    count = len(os.listdir('/content/stocknet-dataset/tweet/raw'))
    print(f"Done — {count} tickers available.")
else:
    count = len(os.listdir('/content/stocknet-dataset/tweet/raw'))
    print(f"Already available — {count} tickers.")

Already available — 87 tickers.


In [6]:
!pip install -q numpy==2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 70.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytorch-forecasting 1.1.1 requires numpy<2.0.0, but you have numpy 2.0.0 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [10]:
from llama_cpp import Llama
import json, re, pandas as pd
from pathlib import Path
from tqdm import tqdm

TWEET_DIR = Path('/content/stocknet-dataset/tweet/raw')
LLM_PATH  = str(ROOT / 'models/llm/Phi-3-mini-4k-instruct-Q4_K_M.gguf')
OUT_CSV   = ROOT / 'data/sentiment_cache/llm_stocknet.csv'

# Resume support — skips already processed rows
if OUT_CSV.exists():
    done_df   = pd.read_csv(OUT_CSV)
    done_keys = set(done_df['ticker'] + '|' + done_df['date'])
    print(f"Resuming — {len(done_df):,} rows already done.")
else:
    done_df, done_keys = pd.DataFrame(), set()

llm = Llama(model_path=LLM_PATH, n_gpu_layers=-1,
            n_ctx=512, verbose=False)

PROMPT = """Rate the financial sentiment of these tweets from -1.0 \
(very bearish) to +1.0 (very bullish). Reply with ONLY a number.
Tweets: {text}
Score:"""

rows, CHECKPOINT_EVERY = [], 500

for ticker_dir in tqdm(sorted(TWEET_DIR.iterdir())):
    ticker = ticker_dir.name
    for day_file in sorted(ticker_dir.iterdir()):
        date = day_file.name
        if f"{ticker}|{date}" in done_keys:
            continue
        # Line-by-line JSON parsing
        texts = []
        for line in day_file.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                tweet = json.loads(line)
                text  = tweet.get('text', '')
                if text:
                    texts.append(text)
            except:
                continue
        combined = " | ".join(texts[:5])[:400] if texts else "no news"
        out   = llm(PROMPT.format(text=combined),
                    max_tokens=8, temperature=0, stop=["\n"])
        raw   = out['choices'][0]['text'].strip()
        match = re.search(r'-?\d+\.?\d*', raw)
        score = float(match.group()) if match else 0.0
        score = max(-1.0, min(1.0, score))
        rows.append({'ticker':ticker, 'date':date, 'llm_score':score})
        if len(rows) % CHECKPOINT_EVERY == 0:
            pd.concat([done_df, pd.DataFrame(rows)]).to_csv(OUT_CSV, index=False)
            print(f"  Checkpoint saved — {len(done_df)+len(rows):,} rows total")


pd.concat([done_df, pd.DataFrame(rows)]).to_csv(OUT_CSV, index=False)
print("LLM inference done.")

llama_context: n_ctx_seq (512) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
  0%|          | 0/87 [00:00<?, ?it/s]

  Checkpoint saved — 500 rows total


  2%|▏         | 2/87 [01:56<1:09:16, 48.90s/it] 

  Checkpoint saved — 1,000 rows total


  6%|▌         | 5/87 [02:44<27:02, 19.79s/it]

...

  Checkpoint saved — 28,500 rows total


 99%|█████████▉| 86/87 [45:40<00:41, 41.87s/it]

  Checkpoint saved — 29,000 rows total


100%|██████████| 87/87 [46:39<00:00, 32.17s/it]

LLM inference done.


In [7]:
llm_dir = ROOT / 'models/llm'
print("Contents of models/llm/:")
for item in llm_dir.iterdir():
    print(f"  {item.name}  —  {item.stat().st_size/1e9:.2f} GB")

Contents of models/llm/:


In [8]:
from huggingface_hub import hf_hub_download

print("Downloading Phi-3-mini Q4_K_M (~2.4GB)...")
hf_hub_download(
    repo_id='bartowski/Phi-3-mini-4k-instruct-GGUF',
    filename='Phi-3-mini-4k-instruct-Q4_K_M.gguf',
    local_dir=str(ROOT / 'models/llm')
)
print("Done.")

Phi-3-mini-4k-instruct-Q4_K_M.gguf:   0%|          | 0.00/2.39G [00:00<?, ?B/s]

Done.


In [9]:
for item in (ROOT / 'models/llm').iterdir():
    print(f"  {item.name}  —  {item.stat().st_size/1e9:.2f} GB")

  .cache  —  0.00 GB
  Phi-3-mini-4k-instruct-Q4_K_M.gguf  —  2.39 GB


In [3]:
import pandas as pd

fb  = pd.read_csv(ROOT / 'data/sentiment_cache/finbert_stocknet.csv')
llm = pd.read_csv(ROOT / 'data/sentiment_cache/llm_stocknet.csv')

merged = fb.merge(llm, on=['ticker','date'], how='inner')
merged['s_finbert'] = merged['pos'] - merged['neg']
merged['s_llm']     = merged['llm_score']

OUT = ROOT / 'data/sentiment_cache/sentiment_merged.csv'
merged[['ticker','date','s_finbert','s_llm','n']].to_csv(OUT, index=False)
print(f"Merged: {len(merged):,} rows saved to sentiment_merged.csv")
print(merged[['ticker','date','s_finbert','s_llm']].head())

Merged: 29,250 rows saved to sentiment_merged.csv
  ticker        date  s_finbert  s_llm
0   AAPL  2014-01-01   0.043564    0.0
1   AAPL  2014-01-02  -0.002903    0.0
2   AAPL  2014-01-03  -0.146391    0.3
3   AAPL  2014-01-04   0.048711    0.0
4   AAPL  2014-01-05  -0.017499    0.8


In [4]:
import pandas as pd

df = pd.read_csv(ROOT / 'data/sentiment_cache/sentiment_merged.csv')
print(f"Rows:    {len(df):,}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Columns: {list(df.columns)}")
print(f"\ns_finbert range: {df['s_finbert'].min():.3f} to {df['s_finbert'].max():.3f}")
print(f"s_llm range:     {df['s_llm'].min():.3f} to {df['s_llm'].max():.3f}")

Rows:    29,250
Tickers: 87
Columns: ['ticker', 'date', 's_finbert', 's_llm', 'n']

s_finbert range: -0.968 to 0.939
s_llm range:     -1.000 to 1.000
